注意力机制：

$$
\tilde{h}_{(t)} = \sum_i \alpha_{(t,i)} y_{(i)}
$$

with  
$$
\alpha_{(t,t)} = \frac{\exp(e_{(t,i)})}{\sum_{i'} \exp(e_{(t,i')})}
$$

and  
$$
e_{(t,i)} =
\begin{cases}
h_{(t)}^\top y_{(i)} & \text{点} \\
h_{(t)}^\top W y_{(i)} & \text{通用} \\
v^\top \tanh(W[h_{(t)}; y_{(i)}]) & \text{合并}
\end{cases}
$$

In [ ]:
import tensorflow_addons as tfa
attention_mechanism = tfa.seq2seq.attention_wrapper.LuongAttention( 
    units, encoder_state, memory_sequence_length=encoder_sequence_length) 
attention_decoder_cell = tfa.seq2seq.attention_wrapper.AttentionWrapper( 
    decoder_cell, attention_mechanism, attention_layer_size=n_units)

位置嵌入：

$$
P_{p,\,2i} = \sin\!\left(\frac{p}{10\,000^{\,2i/d}}\right)
$$

$$
P_{p,\,2i+1} = \cos\!\left(\frac{p}{10\,000^{\,2i/d}}\right)
$$

In [6]:
import tensorflow as tf
import numpy as np
import keras

class PositionalEncoding(keras.layers.Layer): 
    def __init__(self, max_steps, max_dims, dtype=tf.float32, **kwargs): 
        super().__init__(dtype=dtype, **kwargs) 
        if max_dims % 2 == 1: max_dims += 1 # max_dims must be even 
        p, i = np.meshgrid(np.arange(max_steps), np.arange(max_dims // 2)) 
        pos_emb = np.empty((1, max_steps, max_dims)) 
        pos_emb[0, :, ::2] = np.sin(p / 10000 ** (2 * i / max_dims)).T 
        pos_emb[0, :, 1::2] = np.cos(p / 10000 ** (2 * i / max_dims)).T 
        self.positional_embedding = tf.constant(pos_emb.astype(self.dtype)) 
    def call(self, inputs): 
        shape = tf.shape(inputs) 
        return inputs + self.positional_embedding[:, :shape[-2], :shape[-1]] 

embed_size = 512; max_steps = 500; vocab_size = 10000 
encoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32) 
decoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32) 
embeddings = keras.layers.Embedding(vocab_size, embed_size) 
encoder_embeddings = embeddings(encoder_inputs) 
decoder_embeddings = embeddings(decoder_inputs) 
positional_encoding = PositionalEncoding(max_steps, max_dims=embed_size) 
encoder_in = positional_encoding(encoder_embeddings) 
decoder_in = positional_encoding(decoder_embeddings) 

缩放点积注意力

$$
\text{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_{\text{keys}}}}\right)V
$$

在此等式中：

- $ Q $ 是一个矩阵，每个查询包含一行。其形状为 $[n_{\text{queries}},\, d_{\text{keys}}]$，其中 $n_{\text{queries}}$ 是查询数，而 $d_{\text{keys}}$ 是每个查询和每个键的维度。

- $ K $ 是一个矩阵，每个键包含一行。其形状为 $[n_{\text{keys}},\, d_{\text{keys}}]$，其中 $n_{\text{keys}}$ 是键和值的数量。

- $ V $ 是一个矩阵，每个值包含一行。其形状为 $[n_{\text{keys}},\, d_{\text{values}}]$，其中 $d_{\text{values}}$ 是每个值的数量。

- $ QK^\top $ 的形状是 $[n_{\text{queries}},\, n_{\text{keys}}]$：每一对查询/键有一个相似性分数。$\operatorname{softmax}$ 函数的输出具有相同的形状，但所有行的总和为 1。最终输出的形状为 $[n_{\text{queries}},\, d_{\text{values}}]$：每个查询有一行，其中每一行代表了查询结果（值的加权和）。

- 比例因子 $\frac{1}{\sqrt{d_{\text{keys}}}}$ 会缩小相似度分数，以避免 $\operatorname{softmax}$ 函数饱和，这会导致很小的梯度。

- 可以在计算 $\operatorname{softmax}$ 之前，通过将一个非常大的负值加到相应的相似性分数中来屏蔽一些键/值对。这在“掩码多头注意力”层中很有用。

In [7]:
Z = encoder_in 
for N in range(6): 
    Z = keras.layers.Attention(use_scale=True)([Z, Z]) 
encoder_outputs = Z 
Z = decoder_in 
for N in range(6): 
    Z = keras.layers.Attention(use_scale=True, causal=True)([Z, Z]) 
    Z = keras.layers.Attention(use_scale=True)([Z, encoder_outputs]) 
outputs = keras.layers.TimeDistributed( 
    keras.layers.Dense(vocab_size, activation="softmax"))(Z)